In [0]:
%run ./variables.py

In [0]:
# Create catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

# Create schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{LANDING_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SYSTEM_SCHEMA}")

In [0]:
# Landing volume for raw files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{LANDING_SCHEMA}.raw_data")

# System volume for checkpoints
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SYSTEM_SCHEMA}.checkpoints")

print(f"✅ Unity Catalog structure created for {CATALOG_NAME}")

In [0]:
# Clean up existing tables from previous runs
for table in [TABLE_BRONZE_TRANSACTIONS, TABLE_SILVER_TRANSACTIONS, TABLE_SILVER_ALERTS,
              TABLE_SILVER_CUSTOMERS, TABLE_SILVER_MERCHANTS,
              TABLE_GOLD_DAILY_SUMMARY, TABLE_GOLD_SAR_PREFILL]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")

# Clean up existing data in volumes if any
dbutils.fs.rm(LANDING_VOLUME_PATH, recurse=True)
dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)

# Create landing subdirectories
dbutils.fs.mkdirs(TRANSACTIONS_RAW_PATH)
dbutils.fs.mkdirs(CUSTOMERS_RAW_PATH)
dbutils.fs.mkdirs(MERCHANTS_RAW_PATH)

# Run the data generator in bootstrap mode
dbutils.widgets.text("action", "bootstrap")